# Machine Learning Fundamentals
## Unit 04 — Random Forests

**Stage 2: Machine Learning | Beginner level | B2 English**  
**Business scenario:** Melbourne house-price estimation  
**Environment:** Google Colab / Jupyter Notebook  
**Core libraries:** Pandas, Matplotlib, scikit-learn

---

## Business question

In the previous units, we trained a Decision Tree and learned that a tree can overfit the training data. Can we improve predictions by combining many trees instead of relying on only one?

In this lesson, you will train a Random Forest Regressor, compare it with a Decision Tree and a simple baseline, and investigate how the number of trees affects validation performance.

### Dataset continuity

We continue using the supplied `melb_data.csv` file and the same target and features used in the previous units:

- **Target (`y`):** `Price`
- **Features (`X`):** `Rooms`, `Bathroom`, `Landsize`, `Lattitude`, `Longtitude`

The dataset contains historical Melbourne housing records. The results are educational examples, not current market valuations.

> **Instructor note:** This lesson builds on feature/target selection, train/validation splits, MAE, and model complexity. It introduces Random Forests without moving into full hyperparameter tuning or cross-validation yet.

## Learning outcomes

By the end of this notebook, you should be able to:

1. Explain the main idea behind a Random Forest.
2. Describe how Random Forests differ from a single Decision Tree.
3. Train a `RandomForestRegressor` using scikit-learn.
4. Compare models fairly using the same validation set and MAE.
5. Explain what `n_estimators` controls.
6. Inspect feature importance and explain its limitations.
7. Communicate a model comparison without overstating what the results prove.

## Notebook roadmap

1. Why Random Forests?
2. Prepare the dataset
3. Train a Random Forest
4. Compare it with a Decision Tree and baseline
5. Experiment with the number of trees
6. Inspect feature importance
7. Practical challenge and readiness check

**Study routine:** Read the Markdown explanation, run the code cell, inspect the output, and answer the questions before continuing.

# 1. Why Random Forests?

## 1.1 The limitation of one Decision Tree

A Decision Tree makes predictions by following one sequence of feature-based decisions to a leaf. It is easy to visualize and explain, but small changes in the training data can sometimes produce a different tree. A deep tree can also fit details that do not generalize to new examples.

In the previous unit, you compared tree sizes and saw why training error alone is not enough. A natural next step is to combine many trees.

## 1.2 The main idea

A **Random Forest** is an ensemble model: it combines predictions from many Decision Trees.

For regression, each tree predicts a numeric value. The forest typically averages the predictions from its trees to produce the final prediction.

Random Forests introduce randomness in two main ways:

1. **Bootstrap sampling:** Each tree is trained on a sample drawn from the training data, with replacement.
2. **Random feature selection:** At each split, the tree considers a subset of features, depending on the model settings.

These sources of variation help the trees make different errors. Averaging their predictions can reduce variance and often improves generalization compared with one unconstrained tree. It does not guarantee that a Random Forest will always perform better.

## 1.3 Decision Tree vs. Random Forest

| Model | How it predicts | Strength | Limitation |
|---|---|---|---|
| Decision Tree | Uses one tree and returns the value at a leaf. | Relatively easy to visualize and explain. | Can be unstable and may overfit. |
| Random Forest | Combines predictions from many trees. | Often produces more stable predictions. | Harder to explain as a whole and may require more computation. |
| Median baseline | Predicts one constant value for every property. | Simple reference point. | Does not use property features. |

A baseline is useful because a model should be compared with a simple alternative—not only with another complex model.

### Check your understanding

1. What is an ensemble model?
2. Why does a Random Forest use many trees instead of one?
3. What is bootstrap sampling?
4. Does a Random Forest guarantee a lower validation error than every Decision Tree? Why or why not?

# 2. Prepare the Data

We repeat the key data preparation steps so this notebook can run independently. In Google Colab, upload `melb_data.csv` to the session before running the next cell. If the file is in the same folder as the notebook, the path below works as written.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

melbourne_file_path = "melb_data.csv"
melbourne_data = pd.read_csv(melbourne_file_path)

feature_names = ["Rooms", "Bathroom", "Landsize", "Lattitude", "Longtitude"]
target_name = "Price"
required_columns = feature_names + [target_name]

print("Original dataset shape:", melbourne_data.shape)
display(melbourne_data.head())

## 2.1 Select features and target

We use the same five numeric features and `Price` target as in the previous notebooks. For this introductory example, we remove rows that have missing values in these selected columns.

**Limitation:** Dropping rows is a simple teaching choice, not a universal best practice. In real projects, inspect missingness and choose an appropriate strategy.

In [ ]:
model_data = melbourne_data[required_columns].dropna().copy()

X = model_data[feature_names]
y = model_data[target_name]

print(f"Rows retained: {len(model_data):,}")
print(f"Rows removed: {len(melbourne_data) - len(model_data):,}")
print("Feature matrix shape:", X.shape)
print("Target shape:", y.shape)

## 2.2 Create one fixed training/validation split

We use the same split for every model comparison. This makes the comparison fairer because each candidate is evaluated on the same held-out properties.

- The **training set** is used to fit models.
- The **validation set** is used to compare candidate models.
- A final **test set** should be kept separate for an unbiased final evaluation after model selection. We will study this workflow in more detail later.

`random_state=0` makes this split reproducible for the same data and setup.

In [ ]:
train_X, val_X, train_y, val_y = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=0
)

print(f"Training rows: {len(train_X):,}")
print(f"Validation rows: {len(val_X):,}")

# 3. Train a Random Forest

## 3.1 Define and fit the model

Scikit-learn provides `RandomForestRegressor` for predicting numeric targets. The `n_estimators` parameter controls how many trees the forest contains.

We will start with 100 trees. We set `random_state=1` to make the model's randomized behavior reproducible, and `n_jobs=-1` to allow scikit-learn to use available CPU cores for parallel work.

In [ ]:
forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=1,
    n_jobs=-1
)

forest_model.fit(train_X, train_y)

print("Random Forest training complete.")

## 3.2 Generate validation predictions

The model has been fitted using only training data. We now use it to predict prices for validation properties that were not used during fitting.

In [ ]:
forest_predictions = forest_model.predict(val_X)

forest_mae = mean_absolute_error(val_y, forest_predictions)

print(f"Random Forest validation MAE: AUD {forest_mae:,.0f}")

forest_comparison = pd.DataFrame({
    "Actual price": val_y,
    "Predicted price": forest_predictions
})
forest_comparison["Absolute error"] = (
    forest_comparison["Actual price"] - forest_comparison["Predicted price"]
).abs()

display(forest_comparison.head(10).round(0))

## 3.3 Interpret the metric

**Mean Absolute Error (MAE)** is the average absolute difference between actual and predicted prices. It uses the same unit as the target—in this case, Australian dollars.

For example, a validation MAE of AUD 200,000 means the predictions differ from actual prices by AUD 200,000 on average in absolute terms for these validation rows. It does not mean every prediction is wrong by exactly that amount.

The MAE depends on the dataset, features, split, and model settings. Treat it as evidence for comparing models in this experiment, not as a guarantee about future properties.

# 4. Compare the Random Forest with Other Models

To understand whether the forest helps, compare it with:
1. A Decision Tree with `max_leaf_nodes=500`, used as a candidate from the previous lesson.
2. A simple baseline that predicts the median price in the training data for every validation property.

All candidates use the same training and validation split.

In [ ]:
# Decision Tree candidate from the previous lesson
tree_model = DecisionTreeRegressor(
    max_leaf_nodes=500,
    random_state=0
)
tree_model.fit(train_X, train_y)
tree_predictions = tree_model.predict(val_X)
tree_mae = mean_absolute_error(val_y, tree_predictions)

# Baseline: predict the training-set median for every validation row
baseline_value = train_y.median()
baseline_predictions = np.full(len(val_y), baseline_value)
baseline_mae = mean_absolute_error(val_y, baseline_predictions)

model_comparison = pd.DataFrame({
    "Model": ["Training-median baseline", "Decision Tree (500 leaves)", "Random Forest (100 trees)"],
    "Validation MAE (AUD)": [baseline_mae, tree_mae, forest_mae]
}).sort_values("Validation MAE (AUD)")

display(model_comparison.style.format({"Validation MAE (AUD)": "AUD {:,.0f}"}))

## 4.1 Visualize the comparison

Lower MAE indicates smaller average absolute prediction errors on this validation set. The chart helps compare the scale of the errors.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(
    model_comparison["Model"],
    model_comparison["Validation MAE (AUD)"]
)
ax.set_title("Validation MAE by Model")
ax.set_ylabel("Mean Absolute Error (AUD)")
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 4.2 Discuss the result

Use the table and chart to answer:

- Which candidate has the lowest validation MAE?
- How much lower or higher is the Random Forest MAE than the Decision Tree MAE?
- Does the forest improve on the simple baseline?
- Why should we avoid claiming that this result proves Random Forests are always better?

**Instructor note:** Results can differ from reference numbers in online lessons because dataset versions, selected features, train/validation splits, and model parameters may differ. Use the results generated in this notebook as the source for classroom discussion.

# 5. Experiment with the Number of Trees

## 5.1 What does `n_estimators` control?

`n_estimators` is the number of Decision Trees in the forest. More trees can make predictions more stable, but they also increase training time and memory use. Adding trees does not guarantee a large improvement in validation MAE.

We will compare 50, 100, and 200 trees while keeping the split and random seed consistent.

In [ ]:
tree_counts = [50, 100, 200]
tree_count_results = []

for n_trees in tree_counts:
    candidate_forest = RandomForestRegressor(
        n_estimators=n_trees,
        random_state=1,
        n_jobs=-1
    )
    candidate_forest.fit(train_X, train_y)
    candidate_predictions = candidate_forest.predict(val_X)

    tree_count_results.append({
        "n_estimators": n_trees,
        "validation_MAE_AUD": mean_absolute_error(val_y, candidate_predictions)
    })

tree_count_df = pd.DataFrame(tree_count_results)
display(tree_count_df.style.format({"validation_MAE_AUD": "AUD {:,.0f}"}))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(
    tree_count_df["n_estimators"],
    tree_count_df["validation_MAE_AUD"],
    marker="o"
)
ax.set_title("Validation MAE vs. Number of Trees")
ax.set_xlabel("Number of trees (n_estimators)")
ax.set_ylabel("Validation MAE (AUD)")
ax.set_xticks(tree_counts)
ax.grid(True, alpha=0.25)
plt.tight_layout()
plt.show()

### Reflection

1. Did validation MAE improve as the number of trees increased?
2. Was the change large enough to matter in this experiment?
3. What trade-off exists between more trees and computation?
4. Why must we keep the same validation split when comparing the candidates?

Avoid selecting a model by repeatedly changing settings and reporting only the most favorable result. Document the candidates you tried and use a separate test set for final evaluation.

# 6. Inspect Feature Importance

A fitted Random Forest exposes `feature_importances_`, an impurity-based measure that summarizes how much each feature contributed to splits across the trees.

This can be a useful first diagnostic, but it has limitations:
- It is not a causal explanation.
- It can favor features with many possible split points.
- It does not prove that a feature will be important on new data.

Treat feature importance as a clue for further investigation, not as proof of business impact.

In [ ]:
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": forest_model.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(importance_df["feature"], importance_df["importance"])
ax.invert_yaxis()
ax.set_title("Random Forest Feature Importance")
ax.set_xlabel("Impurity-based importance")
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# 7. Practical Activity

Complete the Random Forest lesson and exercise:

- [Kaggle — Intro to Machine Learning](https://www.kaggle.com/code/yazeedalasmar/exercise-random-forests)

## Student challenge

1. Change `max_leaf_nodes` for the Decision Tree and compare it with the Random Forest.
2. Add another `n_estimators` value, such as 300, and record the validation MAE.
3. Identify the strongest candidate in your experiment.
4. Explain one benefit and one limitation of Random Forests.
5. Write a short recommendation with one result, one limitation, and one next step.

## Instructor discussion prompt

Ask learners what evidence they would require before deploying the model in a real estate business. Encourage discussion of unseen test data, cross-validation, data quality, changing market conditions, prediction error costs, and interpretability.